In [ ]:
!pip -q install   "pyzdcf @ git+https://github.com/jecampagne/pyzdcf.git"


In [ ]:
!pip -q install   "nufftacf @ git+https://github.com/jecampagne/nufftacf.git@corrf"

In [3]:
import io, os, contextlib, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyzdcf import pyzdcf as _pyzdcf

from nufftacf import (
    compute_ccf_gaussian_nufft,
    compute_ccf_rectangle_nufft,
    compute_ccf_gaussian_realspace,
)

import nufftacf
print(f"nufftacf : {nufftacf.__version__}")
try:
    import pyzdcf as _pz; print(f"pyzdcf   : {_pz.__version__}")
except AttributeError:
    print("pyzdcf   : (version not exposed)")


nufftacf : 0.1.0
pyzdcf   : (version not exposed)


# CCF for irregularly-sampled signals: nufftacf vs pyZDCF

This notebook demonstrates and compares three approaches to the
**cross-correlation function (CCF)** of two independently-sampled
irregular time series:

| Method | Algorithm | Kernel | Error bars |
|---|---|---|---|
| **nufftacf gaussian** | NUFFT + Wiener-Khinchin | Gaussian (`bin_width`) | — |
| **nufftacf rectangle** | NUFFT + Wiener-Khinchin | Rectangle (`bin_width`) | — |
| **pyZDCF** | Z-transformed DCF (Alexander 1997) | Adaptive binning (`minpts`) | Monte Carlo ± |

Test series mirror the irregularly-sampled STS_SIN* / STS_EXP* series from
`pastas_vs_nufftact.ipynb`, extended to the CCF by introducing a known
time delay τ₀ between two observation networks sampling the same process.

> **Installation note**: install both packages in one `pip` call to let
> the resolver satisfy numpy constraints simultaneously.


In [5]:
import matplotlib as mpl
mpl.rcParams.update({
    'font.size'         : 13,
    'axes.titlesize'    : 14,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 14,
    'xtick.labelsize'   : 14,
    'ytick.labelsize'   : 14,
    'legend.fontsize'   : 12,
    'figure.titlesize'  : 16,
    'figure.titleweight': 'bold',
})

## Helper: run pyZDCF CCF from numpy arrays

pyZDCF requires CSV files as input. This helper wraps the call
transparently so the rest of the notebook works with numpy arrays.


In [6]:
def run_zdcf_ccf(t, x, s, y, minpts=11, num_MC=100, err_frac=0.01):
    """Run pyZDCF cross-correlation between (t, x) and (s, y).

    Parameters
    ----------
    t, x    : times and values of signal 1 (sorted ascending)
    s, y    : times and values of signal 2 (sorted ascending)
    minpts  : minimum pairs per lag bin  (controls lag resolution)
    num_MC  : Monte Carlo runs for error estimation
    err_frac: fractional flux error (synthetic data — small constant)

    Returns
    -------
    DataFrame with columns:
    ['tau', '-sig(tau)', '+sig(tau)', 'dcf', '-err(dcf)', '+err(dcf)', '#bin']
    """
    err_x = np.full(len(t), err_frac * (np.std(x) or 1.0))
    err_y = np.full(len(s), err_frac * (np.std(y) or 1.0))
    input_dir  = tempfile.mkdtemp() + "/"
    output_dir = tempfile.mkdtemp() + "/"
    np.savetxt(input_dir + "lc_x.csv", np.column_stack([t, x, err_x]), delimiter=",")
    np.savetxt(input_dir + "lc_y.csv", np.column_stack([s, y, err_y]), delimiter=",")
    params = dict(
        autocf=False, prefix="zdcf", uniform_sampling=False,
        omit_zero_lags=True, minpts=minpts, num_MC=num_MC,
        lc1_name="lc_x.csv", lc2_name="lc_y.csv",
    )
    with contextlib.redirect_stdout(io.StringIO()):
        result = _pyzdcf(
            input_dir=input_dir, output_dir=output_dir,
            intr=False, sep=",", verbose=False, sparse="auto",
            parameters=params,
        )
    return result


## Test series

Two observation networks X and Y sampling the same underlying process,
with a known time delay **τ₀** between them.

- **x(t)**: observations at irregular times t ⊂ [0, 3650 days] (~43 % kept)
- **y(s)**: same process values, times shifted by τ₀ days (same irregularity structure)
- The CCF should peak at **τ = τ₀** for both SIN and EXP series.

Common reference date is used for t and s so that the τ₀ offset is
preserved in numeric coordinates (see notebook comment).


In [7]:
n_days   = 3650
tau_0    = 60
idx_full = pd.date_range(start="2000-01-01", periods=n_days, freq="D")
ref_date = idx_full[0]
rng      = np.random.default_rng(42)

# --- underlying processes ---
sin_full = np.sin(2 * np.pi * idx_full.dayofyear / 365.25)

alpha    = 10.0
kern_b   = np.exp(-np.arange(200) / alpha)
noise    = rng.standard_normal(n_days + 200)
exp_full = np.convolve(noise, kern_b, mode="valid")[:n_days]
exp_full = (exp_full - exp_full.mean()) / exp_full.std()

# --- observation networks ---
mask_x   = rng.random(n_days) > 0.57    # ~43 % kept
idx_x    = idx_full[mask_x]
idx_y    = idx_x + pd.Timedelta(days=tau_0)

def t_common(idx):
    """Elapsed days from the common reference date (preserves tau_0 offset)."""
    return (idx - ref_date).days.to_numpy(dtype=float)

t_num = t_common(idx_x)
s_num = t_common(idx_y)   # = t_num + tau_0

for name, val in [("SIN", sin_full), ("EXP", exp_full)]:
    print(f"{name}: n_x=n_y={mask_x.sum()}  tau_0={tau_0} d"
          f"  t in [{t_num.min():.0f}, {t_num.max():.0f}]"
          f"  s in [{s_num.min():.0f}, {s_num.max():.0f}]")


SIN: n_x=n_y=1517  tau_0=60 d  t in [2, 3647]  s in [62, 3707]
EXP: n_x=n_y=1517  tau_0=60 d  t in [2, 3647]  s in [62, 3707]


## Computing CCF with all three methods

In [ ]:
lags     = np.arange(1.0, 181.0)
bin_width = 0.5

results = {}
for name, val_full in [("SIN", sin_full), ("EXP", exp_full)]:
    x = val_full[mask_x]
    y = val_full[mask_x]   # same values, observed tau_0 days later via idx_y

    print(f"--- {name} ---")
    c_gn, b_gn = compute_ccf_gaussian_nufft(lags, t_num, x, s_num, y, bin_width=bin_width)
    c_rn, b_rn = compute_ccf_rectangle_nufft(lags, t_num, x, s_num, y, bin_width=bin_width)
    c_gr, b_gr = compute_ccf_gaussian_realspace(lags, t_num, x, s_num, y, bin_width=bin_width)

    print(f"  nufftacf gaussian  peak={c_gn.max():.4f} @ lag={lags[np.argmax(c_gn)]:.0f}")
    print(f"  nufftacf rectangle peak={c_rn.max():.4f} @ lag={lags[np.argmax(c_rn)]:.0f}")
    print(f"  realspace gaussian  peak={c_gr.max():.4f} @ lag={lags[np.argmax(c_gr)]:.0f}")

    print(f"  pyZDCF (minpts=11, MC=100) ...", end=" ", flush=True)
    zdcf = run_zdcf_ccf(t_num, x, s_num, y, minpts=11, num_MC=100)
    idx_z = np.argmax(zdcf["dcf"].to_numpy())
    print(f"peak={zdcf['dcf'].iloc[idx_z]:.4f} @ lag={zdcf['tau'].iloc[idx_z]:.1f}"
          f"  ({len(zdcf)} bins)")

    results[name] = dict(
        c_gn=c_gn, b_gn=b_gn,
        c_rn=c_rn, b_rn=b_rn,
        c_gr=c_gr, b_gr=b_gr,
        zdcf=zdcf,
        x=x, y=y,
    )


## Plots

In [ ]:
def plot_ccf_comparison(name, res, tau_0, lags):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
    fig.suptitle(f"{name} series  (tau_0={tau_0} d)", fontsize=13)

    z     = res["zdcf"]
    tau_z = z["tau"].to_numpy()
    dcf_z = z["dcf"].to_numpy()
    elo   = z["-err(dcf)"].to_numpy()
    ehi   = z["+err(dcf)"].to_numpy()
    n_bins = len(z)

    for ax, title, show_rect in zip(axes,
        ["nufftacf gaussian vs pyZDCF", "nufftacf rectangle vs pyZDCF"],
        [False, True]):

        # pyZDCF avec barres d'erreur
        ax.errorbar(tau_z, dcf_z, yerr=[elo, ehi],
                    fmt="o", ms=6, color="green", elinewidth=0.8, capsize=2,
                    label=f"pyZDCF (minpts=11) — {n_bins} bins", zorder=2)

        # nufftacf
        ax.plot(lags, res["c_gn"], color="blue", lw=2,
                label="nufftacf gaussian", zorder=3)
        if show_rect:
            ax.plot(lags, res["c_rn"], color="orange", lw=2, ls="--",
                    label="nufftacf rectangle", zorder=3)
        else:
            ax.plot(lags, res["c_gr"], color="purple", lw=1.5, ls=":",
                    label="nufftacf realspace (ref)", zorder=3)

        ax.axvline(tau_0, color="k", lw=1.2, ls=":", alpha=0.7,
                   label=f"true tau_0={tau_0} d")
        ax.set_xlabel("Lag [days]")
        ax.set_ylabel("CCF")
        ax.set_title(title)
        ax.set_xlim(0, lags.max())
        ax.set_ylim(-0.5, 1.1)
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
for name, res in results.items():
    plot_ccf_comparison(name, res, tau_0, lags)


## Effect of delay τ₀ on CCF peak quality

EXP series with three delays (20, 60, 120 days) — nufftacf gaussian vs pyZDCF.


In [16]:
lags     = np.arange(1.0, 181.0)
bin_width = 0.5

results_bis = {}
for tau in [20, 60, 120]:
    # rebuild series with this tau
    idx_y2 = idx_x + pd.Timedelta(days=tau)
    s2     = t_common(idx_y2)
    x2     = exp_full[mask_x]
    y2     = exp_full[mask_x]

    c_gn2, _ = compute_ccf_gaussian_nufft(lags, t_num, x2, s2, y2, bin_width=0.5)
    zdcf2     = run_zdcf_ccf(t_num, x2, s2, y2, minpts=11, num_MC=100)
    results_bis[tau] = dict(
        c_gn2=c_gn2,
        zdcf2=zdcf2
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
fig.suptitle(r"CCF peak vs delay $\tau_0$  (EXP series, n~1500)", fontsize=13)

for ax, tau in zip(axes, [20, 60, 120]):

    c_gn2 = results_bis[tau]["c_gn2"]
    z2    = results_bis[tau]["zdcf2"]

    tau_z2= z2["tau"].to_numpy(); dcf_z2=z2["dcf"].to_numpy()
    elo2  = z2["-err(dcf)"].to_numpy(); ehi2=z2["+err(dcf)"].to_numpy()

    ax.errorbar(tau_z2, dcf_z2, yerr=[elo2, ehi2],
                fmt="o", ms=6, color="green", elinewidth=0.8, capsize=2,
                label=f"pyZDCF ({len(z2)} bins)", zorder=2)
    ax.plot(lags, c_gn2, color="blue", lw=2, label="nufftacf gaussian", zorder=3)
    ax.axvline(tau, color="k", lw=1.2, ls=":", label=f"true tau_0={tau}")
    ax.set_title(fr"$\tau_0$={tau} d")
    ax.set_xlabel("Lag [days]")
    if ax is axes[0]: ax.set_ylabel("CCF")
    ax.set_xlim(0, lags.max()); ax.set_ylim(-0.5, 1.1)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
